# JJ Nexus Pro — Alchemist AI training

A reproducible nine-step LoRA workflow for Google Colab. Upload `data/alchemist_training_data.jsonl` from the repository before running the dataset cell. Enable a T4 or better GPU in **Runtime → Change runtime type**.

In [ ]:
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes scipy matplotlib sentencepiece huggingface_hub
import os, random, re, json, math, getpass
import torch
if not torch.cuda.is_available():
    print('GPU required. Go to Runtime → Change runtime type → T4 GPU')
    raise SystemExit('Training is intentionally stopped because this notebook requires a CUDA GPU.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
SEED = 42
random.seed(SEED); torch.manual_seed(SEED)
MODEL_ID = 'meta-llama/Meta-Llama-3.1-8B-Instruct'


In [ ]:
try:
    from huggingface_hub import login
    if not os.environ.get('HF_TOKEN'):
        os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face token (required for Llama 3.1): ')
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('Hugging Face authentication succeeded.')
except Exception as exc:
    raise RuntimeError(f'Hugging Face authentication failed. Accept the Llama 3.1 license and set HF_TOKEN. Details: {exc}') from exc


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ['HF_TOKEN'], use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
DATA_PATH = next((candidate for candidate in ['data/alchemist_training_data.jsonl', 'alchemist_training_data.jsonl'] if os.path.exists(candidate)), None)
if DATA_PATH is None:
    raise FileNotFoundError('Upload data/alchemist_training_data.jsonl from JJ Nexus Pro before running this cell.')
dataset = load_dataset('json', data_files=DATA_PATH, split='train')
if len(dataset) < 2500:
    raise ValueError(f'Expected at least 2,500 training examples, found {len(dataset):,}. Regenerate the JSONL before training.')
required_categories = {'SMC Analysis': 1000, 'COT Data Interpretation': 500, 'Combined SMC + COT': 500, 'MSNR Concepts': 300, 'Risk Management': 200}
category_counts = {category: sum(1 for value in dataset['category'] if value == category) for category in required_categories}
if any(category_counts[category] < minimum for category, minimum in required_categories.items()):
    raise ValueError(f'Category coverage is incomplete: {category_counts}')
print(f'Loaded {DATA_PATH} with category coverage: {category_counts}')
def format_chat(example):
    if not isinstance(example.get('messages'), list): raise ValueError('Every row must contain a messages array')
    return {'text': tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)}
dataset = dataset.map(format_chat)
split = dataset.train_test_split(test_size=0.1, seed=SEED)
train_dataset, test_dataset = split['train'], split['test']
token_lengths = [len(tokenizer.encode(row['text'])) for row in dataset.select(range(min(250, len(dataset))))]
print(f'Total examples: {len(dataset):,} | train: {len(train_dataset):,} | test: {len(test_dataset):,}')
print(f'Average tokens (sample): {sum(token_lengths) / max(1, len(token_lengths)):.0f}')
print(f'Min/max tokens (sample): {min(token_lengths) if token_lengths else 0}/{max(token_lengths) if token_lengths else 0}')
for row in random.sample(list(train_dataset), min(3, len(train_dataset))): print(row['text'][:500], '\\n---')


In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, token=os.environ['HF_TOKEN'], quantization_config=quant_config, device_map='auto')
print(f'Base model loaded: {MODEL_ID}')
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'])
model = get_peft_model(model, lora_config)
print(lora_config)
model.print_trainable_parameters()


In [ ]:
import inspect
from transformers import TrainingArguments
from trl import SFTTrainer
training_kwargs = dict(output_dir='./alchemist-ai-checkpoint', per_device_train_batch_size=2, gradient_accumulation_steps=4, learning_rate=2e-4, warmup_ratio=0.03, lr_scheduler_type='cosine', num_train_epochs=3, fp16=True, logging_steps=10, save_strategy='epoch', report_to='none', seed=SEED)
if 'eval_strategy' in inspect.signature(TrainingArguments.__init__).parameters:
    training_kwargs['eval_strategy'] = 'epoch'
else:
    training_kwargs['evaluation_strategy'] = 'epoch'
training_args = TrainingArguments(**training_kwargs)
trainer_kwargs = dict(model=model, train_dataset=train_dataset, eval_dataset=test_dataset, args=training_args, dataset_text_field='text', max_seq_length=2048)
if 'processing_class' in inspect.signature(SFTTrainer.__init__).parameters:
    trainer_kwargs['processing_class'] = tokenizer
else:
    trainer_kwargs['tokenizer'] = tokenizer
trainer = SFTTrainer(**trainer_kwargs)
print('Effective batch size: 8. LoRA is already attached in Cell 5, so the trainer does not attach a second adapter.')
print('Expected T4 runtime: approximately 4–8 hours for 2,500 examples.')


In [ ]:
import matplotlib.pyplot as plt
training_result = trainer.train()
trainer.save_model('./alchemist-ai-checkpoint')
history = trainer.state.log_history
train_points = [(row.get('step'), row.get('loss')) for row in history if row.get('loss') is not None]
eval_points = [(row.get('step'), row.get('eval_loss')) for row in history if row.get('eval_loss') is not None]
plt.figure(figsize=(10, 4)); plt.plot([p[0] for p in train_points], [p[1] for p in train_points], label='train loss')
if eval_points: plt.plot([p[0] for p in eval_points], [p[1] for p in eval_points], label='eval loss', marker='o')
plt.xlabel('step'); plt.ylabel('loss'); plt.title('Alchemist training curves'); plt.legend(); plt.grid(alpha=.2); plt.show()
plt.savefig('training_loss.png', dpi=160, bbox_inches='tight')
print(f'Final training loss: {train_points[-1][1] if train_points else "not recorded"}')
print('Saved training_loss.png')


In [ ]:
def quality_score(text):
    price_levels = re.findall(r'\b\d+[.,]\d{2,4}\b', text)
    institutional = any(term in text.lower() for term in ['commercial', 'liquidity', 'order block', 'institutional'])
    risk_reward = bool(re.search(r'risk[ /-]?reward|reward[ /-]?to[ /-]?risk|1:\\d', text, re.I))
    confluence = bool(re.search(r'confluence', text, re.I))
    words = len(text.split())
    checks = {'specific_price_levels': len(set(price_levels)) >= 2, 'institutional_logic': institutional, 'risk_reward': risk_reward, 'confluence': confluence, 'over_200_words': words >= 200}
    return {**checks, 'word_count': words, 'score': sum(checks.values())}
model.eval()
quality_results = []
for example in random.sample(list(test_dataset), min(5, len(test_dataset))):
    user_message = next(message['content'] for message in example['messages'] if message['role'] == 'user')
    prompt = tokenizer.apply_chat_template([{'role': 'user', 'content': user_message}], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.inference_mode(): output = model.generate(**inputs, max_new_tokens=400, do_sample=False)
    answer = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    metrics = quality_score(answer); quality_results.append(metrics)
    print('MODEL RESPONSE:\\n', answer, '\\nREFERENCE:\\n', example['messages'][-1]['content'][-1200:])
    print('QUALITY:', metrics, '\\n---')
if quality_results:
    print('QUALITY SUMMARY:', {key: sum(1 for result in quality_results if result[key]) / len(quality_results) for key in ['specific_price_levels', 'institutional_logic', 'risk_reward', 'confluence', 'over_200_words']})
else:
    print('No test examples were available for inference.')


In [ ]:
from peft import AutoPeftModelForCausalLM
if not os.path.isdir('./alchemist-ai-checkpoint'):
    raise FileNotFoundError('Run Cell 7 successfully before merging the LoRA adapter.')
adapter_model = AutoPeftModelForCausalLM.from_pretrained('./alchemist-ai-checkpoint', device_map='auto', torch_dtype=torch.float16)
merged_model = adapter_model.merge_and_unload()
merged_model.save_pretrained('./alchemist-ai-merged', safe_serialization=True)
tokenizer.save_pretrained('./alchemist-ai-merged')
print('Merged model saved to ./alchemist-ai-merged')
REPO_ID = input('Hugging Face Hub repo (USERNAME/alchemist-ai-v1), or press Enter to skip: ').strip()
if REPO_ID:
    merged_model.push_to_hub(REPO_ID, safe_serialization=True)
    tokenizer.push_to_hub(REPO_ID)
    print(f'Uploaded model: https://huggingface.co/{REPO_ID}')
else:
    print('Hub upload skipped; the merged model is available locally.')


In [ ]:
# GGUF / GPTQ deployment notes
!pip -q install -U huggingface_hub
import os, subprocess, sys
if not os.path.isdir('/tmp/llama.cpp'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggerganov/llama.cpp.git', '/tmp/llama.cpp'], check=True)
subprocess.run(['cmake', '-S', '/tmp/llama.cpp', '-B', '/tmp/llama.cpp/build', '-DLLAMA_CURL=OFF'], check=True)
subprocess.run(['cmake', '--build', '/tmp/llama.cpp/build', '--target', 'llama-quantize', '-j', '2'], check=True)
subprocess.run([sys.executable, '/tmp/llama.cpp/convert_hf_to_gguf.py', './alchemist-ai-merged', '--outfile', './alchemist-ai-f16.gguf', '--outtype', 'f16'], check=True)
subprocess.run(['/tmp/llama.cpp/build/bin/llama-quantize', './alchemist-ai-f16.gguf', './alchemist-ai-q4_k_m.gguf', 'Q4_K_M'], check=True)
print('GGUF conversion complete. Use the Q4_K_M file with Ollama or an llama.cpp server. For GPTQ, install a maintained GPTQ tool such as AutoGPTQ/Optimum for the target runtime, use a representative calibration set, and validate perplexity before deployment.')
if os.path.exists('./alchemist-ai-q4_k_m.gguf'): print(f'GGUF size: {os.path.getsize("./alchemist-ai-q4_k_m.gguf") / 1e9:.2f} GB')
